In [ ]:
!pip install fastapi uvicorn nest-asyncio pyngrok transformers accelerate torch

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"  # Small enough for free Colab GPU

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model loaded successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully


In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
import nest_asyncio
from pyngrok import ngrok
import threading

nest_asyncio.apply()

app = FastAPI()

class PromptRequest(BaseModel):
    prompt: str

@app.post("/generate")
async def generate(request: PromptRequest):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a robot task planner. Given a scene description, "
                "respond ONLY with a JSON object like: "
                '{"action": "pick", "target_position": [x, y, z]} '
                "No explanation. JSON only."
            )
        },
        {
            "role": "user",
            "content": request.prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=0.1
        )

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

    return {"response": response}

@app.get("/health")
async def health():
    return {"status": "ok"}

In [ ]:
import subprocess
import threading
import time
import uvicorn
import nest_asyncio

nest_asyncio.apply()

# Start FastAPI in background thread
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_server)
thread.daemon = True
thread.start()

time.sleep(2)  # Wait for server to start

# Download and start cloudflare tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

# Run tunnel and grab the URL
process = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Wait and extract the public URL
time.sleep(5)

import re

output = ""
for _ in range(20):
    line = process.stderr.readline().decode("utf-8")
    output += line
    match = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        print(f"\n YOUR PUBLIC URL: {public_url}\n")
        print(f"Health check: {public_url}/health")
        print(f"Generate endpoint: {public_url}/generate")
        break

INFO:     Started server process [8176]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.



 YOUR PUBLIC URL: https://captain-newfoundland-forth-radiation.trycloudflare.com

Health check: https://captain-newfoundland-forth-radiation.trycloudflare.com/health
Generate endpoint: https://captain-newfoundland-forth-radiation.trycloudflare.com/generate


In [ ]:
import requests

# Replace with your URL printed above
PUBLIC_URL = "https://captain-newfoundland-forth-radiation.trycloudflare.com"

# Test health
health = requests.get(f"{PUBLIC_URL}/health")
print("Health:", health.json())

# Test generation
payload = {
    "prompt": "Scene: red cube at position (0.3, 0.1, 0.5) on a table. Task: pick up the red cube."
}
response = requests.post(f"{PUBLIC_URL}/generate", json=payload)
print("Response:", response.json())

INFO:     34.6.228.160:0 - "GET /health HTTP/1.1" 200 OK
Health: {'status': 'ok'}
INFO:     34.6.228.160:0 - "POST /generate HTTP/1.1" 200 OK
Response: {'response': '{"action": "pick", "target_position": [0.3, 0.1, 0.5]}'}


In [ ]:
%%javascript
function KeepAlive() {
  console.log("Keeping alive...");
  document.querySelector("#top-toolbar > colab-connect-button")
    .shadowRoot.querySelector("div > paper-icon-button")
    .click();
}
setInterval(KeepAlive, 60000);

<IPython.core.display.Javascript object>